In [2]:
import pandas as pd
import numpy as np

fy2022 = "/Users/linhchung/Documents/GitHub/Cleaned-OFLC/data/raw/LCA_Disclosure_Data_FY2022_Q4.xlsx"

df = pd.read_excel(fy2022)

print(df.shape)
print(df.head())

(118645, 96)
          CASE_NUMBER CASE_STATUS RECEIVED_DATE DECISION_DATE  \
0  I-200-22177-312631   Certified    2022-06-26    2022-07-01   
1  I-200-22177-312626   Certified    2022-06-26    2022-07-01   
2  I-200-22177-312625   Certified    2022-06-26    2022-07-01   
3  I-200-22177-312634   Certified    2022-06-26    2022-07-01   
4  I-200-22177-312594   Certified    2022-06-26    2022-07-01   

  ORIGINAL_CERT_DATE VISA_CLASS                               JOB_TITLE  \
0                NaT       H-1B                IT Project Managers - IV   
1                NaT       H-1B         Postdoctoral Research Associate   
2                NaT       H-1B             Network Administrator - III   
3                NaT       H-1B                            Manager JC50   
4                NaT       H-1B  Senior JAVA Developer/Designer Analyst   

     SOC_CODE                                    SOC_TITLE FULL_TIME_POSITION  \
0  15-1199.09      Information Technology Project Managers      

In [3]:
print(df.columns.tolist())

['CASE_NUMBER', 'CASE_STATUS', 'RECEIVED_DATE', 'DECISION_DATE', 'ORIGINAL_CERT_DATE', 'VISA_CLASS', 'JOB_TITLE', 'SOC_CODE', 'SOC_TITLE', 'FULL_TIME_POSITION', 'BEGIN_DATE', 'END_DATE', 'TOTAL_WORKER_POSITIONS', 'NEW_EMPLOYMENT', 'CONTINUED_EMPLOYMENT', 'CHANGE_PREVIOUS_EMPLOYMENT', 'NEW_CONCURRENT_EMPLOYMENT', 'CHANGE_EMPLOYER', 'AMENDED_PETITION', 'EMPLOYER_NAME', 'TRADE_NAME_DBA', 'EMPLOYER_ADDRESS1', 'EMPLOYER_ADDRESS2', 'EMPLOYER_CITY', 'EMPLOYER_STATE', 'EMPLOYER_POSTAL_CODE', 'EMPLOYER_COUNTRY', 'EMPLOYER_PROVINCE', 'EMPLOYER_PHONE', 'EMPLOYER_PHONE_EXT', 'NAICS_CODE', 'EMPLOYER_POC_LAST_NAME', 'EMPLOYER_POC_FIRST_NAME', 'EMPLOYER_POC_MIDDLE_NAME', 'EMPLOYER_POC_JOB_TITLE', 'EMPLOYER_POC_ADDRESS1', 'EMPLOYER_POC_ADDRESS2', 'EMPLOYER_POC_CITY', 'EMPLOYER_POC_STATE', 'EMPLOYER_POC_POSTAL_CODE', 'EMPLOYER_POC_COUNTRY', 'EMPLOYER_POC_PROVINCE', 'EMPLOYER_POC_PHONE', 'EMPLOYER_POC_PHONE_EXT', 'EMPLOYER_POC_EMAIL', 'AGENT_REPRESENTING_EMPLOYER', 'AGENT_ATTORNEY_LAST_NAME', 'AGENT_ATT

In [6]:
import pandas as pd

df = pd.read_excel(fy2022)

df["EMPLOYER_NAME"] = (
    df["EMPLOYER_NAME"]
    .astype("string")
    .str.strip()
    .str.upper()
)

keywords = [
    "NVIDIA",
    "APPLE",
    "MICROSOFT",
    "MICRON",
    "ADVANCED MICRO DEVICES",
    "AMAZON",
    "META",
    "FACEBOOK",
    "ALPHABET",
    "GOOGLE",
    "TESLA"
]

mask = df["EMPLOYER_NAME"].str.contains(
    "|".join(keywords),
    na=False,
    regex=True
)

print(
    df.loc[mask, "EMPLOYER_NAME"]
    .value_counts()
    .to_string()
)

EMPLOYER_NAME
AMAZON.COM SERVICES LLC                              3427
MICROSOFT CORPORATION                                2126
GOOGLE LLC                                           1478
AMAZON WEB SERVICES, INC.                            1128
APPLE INC.                                           1117
META PLATFORMS, INC.                                  807
AMAZON DEVELOPMENT CENTER U.S., INC.                  566
ADVANCED MICRO DEVICES, INC.                          191
TESLA, INC.                                           190
NVIDIA CORPORATION                                    134
AMAZON DATA SERVICES, INC                             110
MICRON TECHNOLOGY, INC.                                74
AMAZON WEB SERVICES INC                                62
AMAZON ADVERTISING LLC                                 45
AMAZON DATA SERVICES, INC.                             39
MICRON SEMICONDUCTOR PRODUCTS, INC.                    12
GOOGLE PUBLIC SECTOR LLC                               11


In [7]:
company_map = {
    "NVIDIA CORPORATION": "NVIDIA",
    "APPLE INC.": "Apple",
    "MICROSOFT CORPORATION": "Microsoft",
    "MICRON TECHNOLOGY, INC.": "Micron",
    "ADVANCED MICRO DEVICES, INC.": "AMD",
    "AMAZON.COM, INC.": "Amazon",
    "AMAZON.COM SERVICES LLC": "Amazon",
    "META PLATFORMS, INC.": "Meta",
    "FACEBOOK, INC.": "Meta",
    "ALPHABET INC.": "Alphabet",
    "GOOGLE LLC": "Alphabet",
    "TESLA, INC.": "Tesla",
    "TESLA MOTORS, INC.": "Tesla"
}

df["Company"] = df["EMPLOYER_NAME"].map(company_map)

df = df[df["Company"].notna()].copy()

print(df["Company"].value_counts())

Company
Amazon       3427
Microsoft    2126
Alphabet     1478
Apple        1117
Meta          810
AMD           191
Tesla         190
NVIDIA        134
Micron         74
Name: count, dtype: int64


In [8]:
df_clean = df.copy()

In [9]:
city_names = {
    "SEATTLE": "Seattle",
    "SAN JOSE": "San Jose",
    "NEW YORK": "New York",
    "AUSTIN": "Austin",
    "DALLAS": "Dallas"
}

# Standardize city names
df["WORKSITE_CITY"] = (
    df["WORKSITE_CITY"]
    .astype("string")
    .str.strip()
    .str.upper()
)

# Create the City column
df["City"] = df["WORKSITE_CITY"].map(city_names)

# Keep only the five selected cities
df = df[df["City"].notna()].copy()

print(df["City"].value_counts())

City
Seattle     1331
Austin       636
New York     533
Dallas        95
San Jose      47
Name: count, dtype: int64


In [10]:
print(df["WAGE_UNIT_OF_PAY"].value_counts())

WAGE_UNIT_OF_PAY
Year    2634
Hour       8
Name: count, dtype: int64


In [16]:
import pandas as pd

# Convert wage values to numbers
df["WAGE_RATE_OF_PAY_FROM"] = pd.to_numeric(
    df["WAGE_RATE_OF_PAY_FROM"],
    errors="coerce"
)

# Standardize wage units
df["WAGE_UNIT_OF_PAY"] = (
    df["WAGE_UNIT_OF_PAY"]
    .astype("string")
    .str.strip()
    .str.upper()
)

# Conversion factors
wage_multiplier = {
    "YEAR": 1,
    "MONTH": 12,
    "BI-WEEKLY": 26,
    "WEEK": 52,
    "HOUR": 2080
}

# Create annual wage column
df["Annual_Wage"] = (
    df["WAGE_RATE_OF_PAY_FROM"]
    * df["WAGE_UNIT_OF_PAY"].map(wage_multiplier)
)

# Check results
print(df[[
    "Company",
    "City",
    "WAGE_RATE_OF_PAY_FROM",
    "WAGE_UNIT_OF_PAY",
    "Annual_Wage"
]].head(10))

      Company      City  WAGE_RATE_OF_PAY_FROM WAGE_UNIT_OF_PAY  Annual_Wage
212    Amazon  New York              105706.00             YEAR    105706.00
245    Amazon   Seattle              104374.00             YEAR    104374.00
264    Amazon    Austin               99195.00             YEAR     99195.00
277      Meta  New York              121000.00             YEAR    121000.00
295     Apple  New York              165000.00             YEAR    165000.00
312    Amazon   Seattle              119350.00             YEAR    119350.00
322  Alphabet    Austin              150000.00             YEAR    150000.00
397    Micron  San Jose              149000.00             YEAR    149000.00
426     Apple   Seattle              200000.00             YEAR    200000.00
432      Meta   Seattle              220684.06             YEAR    220684.06


In [18]:
print("Missing wages:", df["Annual_Wage"].isna().sum())

print(df["Annual_Wage"].describe())

Missing wages: 0
count    2.642000e+03
mean     2.258930e+05
std      3.393385e+06
min      4.268200e+04
25%      1.333085e+05
50%      1.600000e+05
75%      1.840000e+05
max      1.745702e+08
Name: Annual_Wage, dtype: float64


In [20]:
df = df.dropna(subset=["Annual_Wage"]).copy()

df = df[df["Annual_Wage"] > 0].copy()

In [22]:
wage_summary = (
    df
    .groupby(["City", "Company"])
    .agg(
        Number_of_Records=("Annual_Wage", "count"),
        Median_Wage=("Annual_Wage", "median"),
        Mean_Wage=("Annual_Wage", "mean")
    )
    .reset_index()
)

print(wage_summary)

        City    Company  Number_of_Records  Median_Wage      Mean_Wage
0     Austin        AMD                 89    150000.00  150021.479775
1     Austin   Alphabet                 47    146000.00  151864.893617
2     Austin     Amazon                231    150000.00  148896.030303
3     Austin      Apple                163    160000.00  157426.147239
4     Austin       Meta                 16    165238.89  167919.351875
5     Austin  Microsoft                 37    150000.00  152917.162162
6     Austin     NVIDIA                 10    155972.00  156800.200000
7     Austin      Tesla                 43    140000.00  138393.042791
8     Dallas     Amazon                 77    140000.00  141019.064935
9     Dallas       Meta                  2    124000.00  124000.000000
10    Dallas  Microsoft                 16    152000.00  144750.375000
11  New York        AMD                  1    177081.00  177081.000000
12  New York   Alphabet                141    176000.00  179015.414184
13  Ne

In [26]:
columns_to_keep = [
    "Company",
    "EMPLOYER_NAME",
    "JOB_TITLE",
    "SOC_TITLE",
    "City",
    "WAGE_RATE_OF_PAY_FROM",
    "WAGE_UNIT_OF_PAY",
    "Annual_Wage"
]S

df_clean = df[columns_to_keep].copy()

df_clean.to_csv(
    "OFLC_BigTech_Wages_2022.csv",
    index=False
)